In [1]:
import numpy as np
np.random.seed(42)
X=np.random.uniform(-2,2,(400,3))
y=(np.sin(X[:,0])+0.5*(X[:,1]**2)-0.8*X[:,2])
y=y.reshape(-1,1)


In [2]:
def relu(z):return np.maximum(0,z)
def relu_d(z):return (z>0).astype(float)

def sigmoid(z):return 1/(1+np.exp(-z))
def sigmoid_d(z):s=sigmoid(z);return s*(1-s)

def tanh(z):return np.tanh(z)
def tanh_d(z):t=np.tanh(z);return 1-t**2

def leakyrelu(z,a=0.01):return np.where(z>0,z,a*z)
def leakyrelu_d(z,a=0.01):g=np.ones_like(z);g[z<=0]=a;return g

def softplus(z):return np.log(1+np.exp(z))
def softplus_d(z):return sigmoid(z)


In [3]:
def init(layers):
    W=[]
    b=[]
    for i in range(len(layers)-1):
        W.append(np.random.uniform(-0.5,0.5,(layers[i+1],layers[i])))
        b.append(np.zeros((layers[i+1],1)))
    return W,b


In [4]:
def forward(X,W,b,act):
    A=X.T
    Zs=[]
    As=[A]
    for i in range(len(W)-1):
        Z=W[i]@A+b[i]
        A=act(Z)
        Zs.append(Z)
        As.append(A)
    Z=W[-1]@A+b[-1]
    A=Z
    Zs.append(Z)
    As.append(A)
    return Zs,As


In [5]:
def backward(X,y,W,b,Zs,As,act_d):
    m=X.shape[0]
    dWs=[None]*len(W)
    dbs=[None]*len(b)
    yhat=As[-1]
    dA=2*(yhat-y.T)
    for i in reversed(range(len(W))):
        if i==len(W)-1:
            dZ=dA
        else:
            dZ=dA*act_d(Zs[i])
        dW=(1/m)*(dZ@As[i].T)
        db=(1/m)*np.sum(dZ,axis=1,keepdims=True)
        dA=W[i].T@dZ
        dWs[i]=dW
        dbs[i]=db
    return dWs,dbs


In [6]:
def train(layers,act,act_d,lr=0.01,epochs=1000):
    W,b=init(layers)
    for e in range(epochs):
        Zs,As=forward(X,W,b,act)
        dWs,dbs=backward(X,y,W,b,Zs,As,act_d)
        for i in range(len(W)):
            W[i]=W[i]-lr*dWs[i]
            b[i]=b[i]-lr*dbs[i]
    loss=np.mean((As[-1].T-y)**2)
    g1=np.linalg.norm(dWs[0])
    gl=np.linalg.norm(dWs[-2]) if len(W)>1 else np.linalg.norm(dWs[0])
    return loss,g1,gl


In [7]:
models={
"A":[3,4,1],
"B":[3,6,6,1],
"C":[3,8,8,8,8,1],
"D":[3,8,8,8,8,8,8,8,8,1]
}

for name,l in models.items():
    loss,g1,gl=train(l,relu,relu_d)
    print("Model",name,"ReLU",loss,g1,gl)

for name,l in models.items():
    loss,g1,gl=train(l,sigmoid,sigmoid_d)
    print("Model",name,"Sigmoid",loss,g1,gl)


Model A ReLU 0.11154512827725771 0.045217360748741345 0.045217360748741345
Model B ReLU 0.07288586596130892 0.03660874595282526 0.02144080423845441
Model C ReLU 0.030451498206553077 0.02387634547844159 0.01680066614352688
Model D ReLU 0.0531848271858129 0.42978396573665373 0.6212899111786582
Model A Sigmoid 0.4034798512536561 0.02328218387802834 0.02328218387802834
Model B Sigmoid 0.6847471250676317 0.20507396604773612 0.2467119550683771
Model C Sigmoid 1.7431256770560626 0.001908655651767695 0.0029956420404407646
Model D Sigmoid 1.743852977779307 2.517545785570898e-06 2.031595837047436e-06
